# VolleyVision: TrackNet round 2 (old set + your checked clips)

Continues from the current fine-tuned TrackNet (`FT_TrackNet_best.pt`) on a new export that contains the original
training set **plus** your checked clips:

    python -m tools.export_training --name v2 --tracknet-base training/tracknet/datasets/tracknet_v1 --archive
    # optional, train at your clips' shape (less squeezed than 512x288):  add  --tracknet-size 640x288

Your 10 test clips are never in the export.

**Put in Google Drive under `MyDrive/VolleyVision/`:**
- `tracknet_v2.tar` (from `training/tracknet/datasets/`)
- `FT_TrackNet_best.pt` (from `models/tracknet/`)
- `patch_tracknet.py` (from `training/tracknet/`, the updated one)

Runtime → Change runtime type → **GPU**, then Run all. Checkpoints go to `MyDrive/VolleyVision/tracknet_ft_v2/` every epoch.


In [ ]:
# ---- settings ----
DATA = 'tracknet_v2'                 # the tar's name without .tar
INIT = 'FT_TrackNet_best.pt'         # start from this checkpoint (in MyDrive/VolleyVision)
INPUT_WIDTH = 512                    # 512 = as before; 640 = your clips' 2.17:1 shape (288 high either way)
EPOCHS = 15
OUT = 'tracknet_ft_v2'

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/VolleyVision'
import os
for f in [f'{DATA}.tar', INIT, 'patch_tracknet.py']:
    print(('OK      ' if os.path.exists(f'{D}/{f}') else 'MISSING ') + f)

In [ ]:
%cd /content
!rm -rf TrackNetV3 && git clone -q https://github.com/qaz812345/TrackNetV3.git && cd TrackNetV3 && git checkout -q 6eda442
!pip install -q parse pycocotools
!mkdir -p /content/data && tar -xf {D}/{DATA}.tar -C /content/data
os.environ['TN_HEIGHT'], os.environ['TN_WIDTH'] = '288', str(INPUT_WIDTH)
!python {D}/patch_tracknet.py /content/TrackNetV3 /content/data/{DATA}
!grep -n "^HEIGHT\|^WIDTH" /content/TrackNetV3/utils/general.py
!head -c 600 /content/data/{DATA}/manifest.json

## Train
Learning rate 1e-4 (fine-tuning). At 640 wide each step is ~25% slower than at 512.
`TrackNet_best.pt` = best validation epoch; the input size is stored in the checkpoint, so the pipeline runs it at the
right size automatically.

In [ ]:
%cd /content/TrackNetV3
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['INIT_CKPT'] = f'{D}/{INIT}'
!python train.py --model_name TrackNet --seq_len 8 --bg_mode concat --alpha 0.5 \
    --epochs {EPOCHS} --batch_size 10 --learning_rate 1e-4 --save_dir {D}/{OUT} --verbose

## Done
Copy `MyDrive/VolleyVision/tracknet_ft_v2/TrackNet_best.pt` to `models/tracknet/` on the Mac with a new name
(e.g. `FT2_TrackNet_best.pt`), then:

    python -m tools.scoreboard --tracknet models/tracknet/FT2_TrackNet_best.pt     # vs the current TrackNet
    python -m tools.tracknet_offset --ckpt models/tracknet/FT2_TrackNet_best.pt --check   # is the offset gone?


## If the runtime disconnected
Re-run the setup cells (everything above **Train**), then this cell.

In [ ]:
%cd /content/TrackNetV3
!python train.py --model_name TrackNet --epochs {EPOCHS} --save_dir {D}/{OUT} --resume_training --verbose